# Qualification des données — Olist

**Objectif** : vérifier que le jeu de données permet de répondre à la
problématique définie dans la note de cadrage v1.

**Méthode** : chaque hypothèse est énoncée, testée par un contrôle chiffré,
puis validée ou invalidée avec sa conséquence sur le périmètre.

In [1]:
import pandas as pd

chemin = "dataset/"

orders    = pd.read_csv(chemin + "olist_orders_dataset.csv")
customers = pd.read_csv(chemin + "olist_customers_dataset.csv")
items     = pd.read_csv(chemin + "olist_order_items_dataset.csv")
reviews   = pd.read_csv(chemin + "olist_order_reviews_dataset.csv")
payments  = pd.read_csv(chemin + "olist_order_payments_dataset.csv")

for nom, table in [("orders", orders), ("customers", customers),
                   ("items", items), ("reviews", reviews), ("payments", payments)]:
    print(f"{nom:12} {table.shape}")

orders       (99441, 8)
customers    (99441, 5)
items        (112650, 7)
reviews      (99224, 7)
payments     (103886, 5)


## H1 — Volume suffisant de clients multi-commandes

**Hypothèse** : un nombre significatif de clients a passé plusieurs commandes,
permettant d'observer l'évolution du comportement d'achat après un retard.

**Contrôle** : nombre de `customer_unique_id` associés à plus d'une commande.

**Point de vigilance** : `customer_id` change à chaque commande ; seul
`customer_unique_id` identifie une personne.

In [2]:
df = orders.merge(customers, on="customer_id", how="left")
print("Contrôle jointure :", orders.shape[0], "→", df.shape[0])

commandes_par_client = df.groupby("customer_unique_id")["order_id"].count()

print("Clients uniques         :", len(commandes_par_client))
print("Clients multi-commandes :", (commandes_par_client > 1).sum())
print("Taux de réachat         :", round((commandes_par_client > 1).mean() * 100, 2), "%")
print()
print(commandes_par_client.value_counts().head())

Contrôle jointure : 99441 → 99441
Clients uniques         : 96096
Clients multi-commandes : 2997
Taux de réachat         : 3.12 %

order_id
1    93099
2     2745
3      203
4       30
5        8
Name: count, dtype: int64


**Résultat** : 96 096 clients uniques, dont 2 997 avec plusieurs commandes
(3,12 %). 93 099 clients n'ont passé qu'une seule commande.

**Conclusion** : hypothèse **invalidée**. Aucune trajectoire client n'est
observable. L'analyse du comportement d'achat après un retard est abandonnée ;
le chiffrage de la valeur est reporté au grain vendeur.

## H2 — Dates de livraison disponibles

**Hypothèse** : les dates de livraison estimée et réelle sont renseignées sur
la majorité des commandes livrées.

**Contrôle** : taux de valeurs manquantes sur les commandes au statut
`delivered`.

In [3]:
print(orders["order_status"].value_counts())
print()

livrees = orders[orders["order_status"] == "delivered"]

print("Commandes livrées                :", len(livrees))
print("Sans date de livraison réelle    :", livrees["order_delivered_customer_date"].isna().sum())
print("Sans date estimée                :", livrees["order_estimated_delivery_date"].isna().sum())
print("Sans date de remise transporteur :", livrees["order_delivered_carrier_date"].isna().sum())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Commandes livrées                : 96478
Sans date de livraison réelle    : 8
Sans date estimée                : 0
Sans date de remise transporteur : 2


**Résultat** : 96 478 commandes livrées. 8 sans date de livraison réelle,
0 sans date estimée, 2 sans date de remise au transporteur.

**Conclusion** : hypothèse **validée** (99,99 %). Périmètre d'analyse fixé à
96 470 commandes. La décomposition du délai en trois segments est exploitable
sur la quasi-totalité du périmètre.

## H3 — Note de satisfaction disponible

**Hypothèse** : une part importante des commandes livrées dispose d'une note
de satisfaction.

**Contrôle** : couverture des avis sur les commandes livrées.

In [4]:
print("Lignes dans reviews          :", len(reviews))
print("Commandes distinctes évaluées :", reviews["order_id"].nunique())
print()

couverture = livrees["order_id"].isin(reviews["order_id"]).mean()
print("Part des commandes livrées avec avis :", round(couverture * 100, 2), "%")

Lignes dans reviews          : 99224
Commandes distinctes évaluées : 98673

Part des commandes livrées avec avis : 99.33 %


**Résultat** : 99,33 % des commandes livrées disposent d'un avis.
99 224 lignes pour 98 673 commandes distinctes : environ 550 commandes ont
plusieurs évaluations.

**Conclusion** : hypothèse **validée**. Le lien retard-satisfaction est
mesurable sans biais de couverture. Une règle de sélection devra être définie
pour les commandes à évaluations multiples.

## H5 — Attribution du retard à un vendeur

**Hypothèse** : la majorité des commandes ne comporte qu'un seul vendeur,
permettant d'attribuer un retard à un responsable identifié.

**Contrôle** : distribution du nombre de vendeurs distincts par commande.

In [5]:
vendeurs_par_commande = items.groupby("order_id")["seller_id"].nunique()

print(vendeurs_par_commande.value_counts())
print("Part mono-vendeur :", round((vendeurs_par_commande == 1).mean() * 100, 2), "%")
print()
print("Vendeurs distincts :", items["seller_id"].nunique())

seller_id
1    97388
2     1219
3       54
4        3
5        2
Name: count, dtype: int64
Part mono-vendeur : 98.7 %

Vendeurs distincts : 3095


**Résultat** : 98,7 % des commandes sont mono-vendeur (97 388 sur 98 666).
3 095 vendeurs distincts.

**Conclusion** : hypothèse **validée**. L'attribution du retard à un vendeur
est possible. Les 1,3 % de commandes multi-vendeurs sont écartées de l'analyse
au grain vendeur uniquement.

## H7 — Cohérence chronologique des dates

**Hypothèse** : les dates respectent l'ordre logique du parcours :
achat ≤ approbation ≤ remise transporteur ≤ livraison.

**Contrôle** : comptage des violations de chaque étape.

In [6]:
colonnes_dates = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
for col in colonnes_dates:
    orders[col] = pd.to_datetime(orders[col])

livrees = orders[orders["order_status"] == "delivered"]

print("Approbation avant achat            :", (livrees["order_approved_at"] < livrees["order_purchase_timestamp"]).sum())
print("Remise transporteur avant approbation :", (livrees["order_delivered_carrier_date"] < livrees["order_approved_at"]).sum())
print("Livraison avant remise transporteur :", (livrees["order_delivered_customer_date"] < livrees["order_delivered_carrier_date"]).sum())

Approbation avant achat            : 0
Remise transporteur avant approbation : 1350
Livraison avant remise transporteur : 23


**Résultat** : 0 approbation antérieure à l'achat, 1 350 remises au
transporteur antérieures à l'approbation (1,4 %), 23 livraisons antérieures à
la remise au transporteur.

**Conclusion** : hypothèse **partiellement validée**. Deux règles de gestion
à définir : exclusion des 23 commandes chronologiquement impossibles, et
traitement des 1 350 commandes expédiées avant validation du paiement, qui
faussent le calcul du segment « préparation vendeur ».

## H9 — Reconstitution du montant des commandes

**Hypothèse** : le montant d'une commande est reconstituable de manière fiable.

**Contrôle** : écart entre la somme des articles (prix + fret) et la somme des
paiements enregistrés.

In [ ]:
montant_items = items.groupby("order_id")[["price", "freight_value"]].sum()
montant_items["total_items"] = montant_items["price"] + montant_items["freight_value"]

montant_paiements = payments.groupby("order_id")["payment_value"].sum()

comparaison = montant_items[["total_items"]].join(montant_paiements)
comparaison["ecart"] = (comparaison["total_items"] - comparaison["payment_value"]).abs()

print(comparaison["ecart"].describe())
print()
print("Commandes avec écart > 1 R$ :", (comparaison["ecart"] > 1).sum())

count    98665.000000
mean         0.033162
std          1.129109
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max        182.810000
Name: ecart, dtype: float64

Commandes avec écart > 1 R$ : 249


**Résultat** : médiane et 75e centile de l'écart à zéro. 249 commandes
présentent un écart supérieur à 1 R$, soit 0,25 % du périmètre.

**Conclusion** : hypothèse **validée**. Le chiffrage financier est fiable.
Les écarts résiduels sont documentés sans impact sur les conclusions.

## H11 — Historique suffisant par vendeur

**Hypothèse** : les vendeurs disposent d'un volume de commandes permettant
d'observer une trajectoire dans le temps.

**Contrôle** : distribution du nombre de commandes par vendeur, et part du
volume total couverte selon différents seuils.

In [8]:
commandes_par_vendeur = items.groupby("seller_id")["order_id"].nunique()

print(commandes_par_vendeur.describe())
print()

for seuil in [10, 30, 50, 100]:
    n = (commandes_par_vendeur >= seuil).sum()
    part = commandes_par_vendeur[commandes_par_vendeur >= seuil].sum() / commandes_par_vendeur.sum()
    print(f"Vendeurs avec >= {seuil:3} commandes : {n:5} ({round(part*100,1)} % du volume)")

count    3095.000000
mean       32.313409
std       105.139763
min         1.000000
25%         2.000000
50%         6.000000
75%        21.500000
max      1854.000000
Name: order_id, dtype: float64

Vendeurs avec >=  10 commandes :  1271 (93.9 % du volume)
Vendeurs avec >=  30 commandes :   634 (83.3 % du volume)
Vendeurs avec >=  50 commandes :   429 (75.5 % du volume)
Vendeurs avec >= 100 commandes :   210 (60.0 % du volume)


**Résultat** : distribution très concentrée — moyenne de 32 commandes pour une
médiane de 6, maximum à 1 854. Un quart des vendeurs a 2 commandes ou moins.

Avec un seuil de 10 commandes minimum : 1 271 vendeurs retenus, couvrant
93,9 % du volume total.

**Conclusion** : hypothèse **validée sous condition**. Un seuil de 10 commandes
est retenu comme règle de gestion pour l'analyse au grain vendeur : il élimine
les vendeurs non analysables tout en préservant 93,9 % de l'activité.

---

## Synthèse de la qualification

| # | Hypothèse | Résultat | Conséquence |
|---|---|---|---|
| H1 | Réachat client suffisant | ❌ 3,12 % | Analyse avant/après abandonnée |
| H2 | Dates disponibles | ✅ 99,99 % | Périmètre : 96 470 commandes |
| H3 | Avis disponibles | ✅ 99,33 % | Lien retard-satisfaction mesurable |
| H5 | Commandes mono-vendeur | ✅ 98,7 % | Attribution au vendeur possible |
| H6 | Date d'annulation | ❌ inexistante | Piste du délai de patience abandonnée |
| H7 | Cohérence des dates | ⚠️ 1 373 anomalies | Deux règles de gestion à définir |
| H9 | Montant reconstituable | ✅ 99,75 % | Chiffrage fiable |
| H11 | Historique vendeur | ✅ sous condition | Seuil de 10 commandes retenu |

### Conclusion générale

La donnée permet de répondre à la problématique, à l'exception de la mesure de
l'impact du retard sur la fidélité client. Le chiffrage de la valeur est
reporté au grain vendeur, pour lequel un historique exploitable existe.

Le périmètre révisé est formalisé dans la note de cadrage v2.

## Contrôles complémentaires — couverture des tables de référence

In [4]:
import pandas as pd 

chemin = "dataset/"
translation = pd.read_csv(chemin + "product_category_name_translation.csv")
products = pd.read_csv(chemin + "olist_products_dataset.csv")

print("Catégories dans products    :", products["product_category_name"].nunique())
print("Catégories traduites        :", len(translation))

Catégories dans products    : 73
Catégories traduites        : 71


**Résultat** : 73 catégories dans `products`, 71 traduites. Deux catégories
sans correspondance.

**Conséquence** : jointure `left` avec conservation du nom portugais par défaut.